In [2]:
import os
import kagglehub

path = kagglehub.dataset_download("chiranjivdas09/ta-feng-grocery-dataset")

print("Path to dataset files:", path)

for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Path to dataset files: /home/parth/.cache/kagglehub/datasets/chiranjivdas09/ta-feng-grocery-dataset/versions/1
/home/parth/.cache/kagglehub/datasets/chiranjivdas09/ta-feng-grocery-dataset/versions/1/ta_feng_all_months_merged.csv


In [3]:
import pandas as pd
import numpy as np

print("Loading Ta Feng data...")

df = pd.read_csv(os.path.join(path, 'ta_feng_all_months_merged.csv'))

df = df[['CUSTOMER_ID', 'TRANSACTION_DT', 'PRODUCT_ID']]

df = df.drop_duplicates()

print(f"Total unique transactions (baskets): {df.groupby(['CUSTOMER_ID', 'TRANSACTION_DT']).ngroups}")
print(f"Total unique users: {df['CUSTOMER_ID'].nunique()}")
print(f"Total unique items: {df['PRODUCT_ID'].nunique()}")

Loading Ta Feng data...
Total unique transactions (baskets): 119578
Total unique users: 32266
Total unique items: 23812


Since Ta Feng's dataset contains over 23,000 (23, 812 to be exact) unique items, most of which rarely appear
Truncate the dataset for top 1,000 items

In [4]:
print("Calculating item frequencies...")
item_counts = df['PRODUCT_ID'].value_counts()


TOP_K = 1000
top_items = set(item_counts.head(TOP_K).index)

print(f"Top {TOP_K} items capture {item_counts.head(TOP_K).sum() / len(df):.2%} of all item interactions.")


print("Mapping rare items...")
df['ITEM_MAPPED'] = df['PRODUCT_ID'].apply(lambda x: x if x in top_items else 0)


df['TRANSACTION_DT'] = pd.to_datetime(df['TRANSACTION_DT'])
df = df.sort_values(by=['CUSTOMER_ID', 'TRANSACTION_DT'])


print("Data mapped and sorted chronologically.")

Calculating item frequencies...
Top 1000 items capture 45.89% of all item interactions.
Mapping rare items...
Data mapped and sorted chronologically.


Transform the raw product id into contiguous indicies(0 indexed) for the DL models to understand

In [5]:
unique_items = [0] + list(top_items) 
item_to_idx = {item: idx for idx, item in enumerate(unique_items)}


print("Mapping raw IDs to contiguous indices (0-1000)...")
df['ITEM_IDX'] = df['ITEM_MAPPED'].map(item_to_idx)

Mapping raw IDs to contiguous indices (0-1000)...


Create customer baskets by grouping the data of one customer present in multiple rows for the same day into a single list of items

In [6]:
print("Grouping items into individual baskets...")

baskets = df.groupby(['CUSTOMER_ID', 'TRANSACTION_DT'])['ITEM_IDX'].apply(list).reset_index()


baskets = baskets.sort_values(['CUSTOMER_ID', 'TRANSACTION_DT'])

print(f"Total baskets formed: {len(baskets)}")
print(baskets.head())

Grouping items into individual baskets...
Total baskets formed: 119578
   CUSTOMER_ID TRANSACTION_DT              ITEM_IDX
0         1069     2000-11-13              [430, 0]
1         1069     2001-01-21           [0, 103, 0]
2         1069     2001-02-03  [62, 0, 0, 896, 237]
3         1069     2001-02-10                 [559]
4         1113     2000-11-12              [0, 970]


Group individual baskets based on cutomer_id
We need atleast three baskets per user; 2 for training and 1 for target prediction

In [7]:
print("Grouping baskets into user sequences...")

user_sequences = baskets.groupby('CUSTOMER_ID')['ITEM_IDX'].apply(list).reset_index()
user_sequences.rename(columns={'ITEM_IDX': 'BASKET_SEQUENCE'}, inplace=True)


min_baskets = 3
user_sequences = user_sequences[user_sequences['BASKET_SEQUENCE'].apply(len) >= min_baskets]

print(f"Users retained (>= {min_baskets} baskets): {len(user_sequences)}")


sample_seq = user_sequences.iloc[0]['BASKET_SEQUENCE']
print(f"\nSample user sequence length (number of trips): {len(sample_seq)}")
print(f"Trip 1 items: {sample_seq[0]}")
print(f"Trip 2 items: {sample_seq[1]}")

Grouping baskets into user sequences...
Users retained (>= 3 baskets): 14074

Sample user sequence length (number of trips): 4
Trip 1 items: [430, 0]
Trip 2 items: [0, 103, 0]


/tmp/ipykernel_109293/722820702.py:4: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  user_sequences.rename(columns={'ITEM_IDX': 'BASKET_SEQUENCE'}, inplace=True)


Multi hot encoding

In [8]:

VOCAB_SIZE = 1001 

def multi_hot_encode(basket, vocab_size):
    vector = np.zeros(vocab_size, dtype=np.float32)
    for item_idx in basket:
        vector[item_idx] = 1.0
    return vector

print("Multi-hot encoding all baskets...")

user_sequences['MULTI_HOT_SEQ'] = user_sequences['BASKET_SEQUENCE'].apply(
    lambda seq: [multi_hot_encode(basket, VOCAB_SIZE) for basket in seq]
)

print(f"Sample multi-hot vector shape: {user_sequences['MULTI_HOT_SEQ'].iloc[0][0].shape}")

Multi-hot encoding all baskets...
Sample multi-hot vector shape: (1001,)


Split the baskets into training and test

In [9]:
print("Splitting sequences into historical inputs (X) and target baskets (y)...")

user_sequences['X_SEQ'] = user_sequences['MULTI_HOT_SEQ'].apply(lambda seq: seq[:-1])

user_sequences['Y_TARGET'] = user_sequences['MULTI_HOT_SEQ'].apply(lambda seq: seq[-1])

X_raw = np.vstack(user_sequences['MULTI_HOT_SEQ'].sum(axis=0))
X_list = user_sequences['X_SEQ'].tolist()
y_array = np.array(user_sequences['Y_TARGET'].tolist(), dtype=np.float32)

print(f"Target array (y) shape: {y_array.shape}")
print(f"Raw sequences array (X) shape: {X_raw.shape}")

Splitting sequences into historical inputs (X) and target baskets (y)...
Target array (y) shape: (14074, 1001)
Raw sequences array (X) shape: (95072, 1001)


Padding user sequences to fit into Bi-LSTM model

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os


MAX_SEQ_LEN = 10 

print(f"Padding sequences to length {MAX_SEQ_LEN}...")

X_padded = pad_sequences(
    X_list, 
    maxlen = MAX_SEQ_LEN, 
    padding = 'pre', 
    dtype = 'float32',
    value = 0.0 
)

print(f"Final Input (X) shape: {X_padded.shape}")
print(f"Final Target (y) shape: {y_array.shape}")


os.makedirs('../data/processed', exist_ok=True)

print("Saving tensors to disk...")
np.save('../data/processed/X_raw.npy', X_raw)
np.save('../data/processed/X_padded.npy', X_padded)
np.save('../data/processed/y_target.npy', y_array)


I0000 00:00:1793599916.391088  109293 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793599916.417554  109293 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793599917.220177  109293 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Padding sequences to length 10...
Final Input (X) shape: (14074, 10, 1001)
Final Target (y) shape: (14074, 1001)
Saving tensors to disk...
